In [ ]:
from google.cloud import aiplatform

aiplatform.init(
    project="YOUR_PROJECT_ID",
    location="us-central1",
)

In [ ]:
from kfp.v2.dsl import component

@component(
    base_image="python:3.10",
    packages_to_install=[
        "pandas",
        "scikit-learn",
        "google-cloud-aiplatform"
    ],
)
def train_random_forest():
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    from google.cloud import aiplatform

    aiplatform.start_run("rf_run")

    df = pd.read_csv("gs://YOUR_BUCKET/data.csv")

    X = df.drop("label", axis=1)
    y = df["label"]

    model = RandomForestClassifier(n_estimators=100)
    model.fit(X, y)

    acc = model.score(X, y)

    aiplatform.log_metrics({"accuracy": acc})

    aiplatform.end_run()

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas","xgboost","google-cloud-aiplatform"],
)
def train_xgboost():
    import pandas as pd
    import xgboost as xgb
    from google.cloud import aiplatform

    aiplatform.start_run("xgb_run")

    df = pd.read_csv("gs://YOUR_BUCKET/data.csv")

    X = df.drop("label", axis=1)
    y = df["label"]

    model = xgb.XGBClassifier()
    model.fit(X, y)

    acc = model.score(X, y)

    aiplatform.log_metrics({"accuracy": acc})

    aiplatform.end_run()

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas","tensorflow","google-cloud-aiplatform"],
)
def train_neural_net():
    import pandas as pd
    import tensorflow as tf
    from google.cloud import aiplatform

    aiplatform.start_run("nn_run")

    df = pd.read_csv("gs://YOUR_BUCKET/data.csv")

    X = df.drop("label", axis=1)
    y = df["label"]

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(X, y, epochs=5)

    acc = model.evaluate(X, y)[1]

    model.upload(
    display_name="customer-churn-model",
)

    aiplatform.log_metrics({"accuracy": acc})

    aiplatform.end_run()

In [ ]:
from kfp.v2.dsl import pipeline

@pipeline(
    name="parallel-training-pipeline"
)
def training_pipeline():

    rf_task = train_random_forest()
    xgb_task = train_xgboost()
    nn_task = train_neural_net()

In [ ]:
from kfp.v2 import compiler

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path="pipeline.json",
)

In [ ]:
from google.cloud import aiplatform

job = aiplatform.PipelineJob(
    display_name="parallel-training",
    template_path="pipeline.json",
)

job.run()

In [ ]:
aiplatform.init(
    project="YOUR_PROJECT_ID",
    location="asia-southeast1",
    experiment="model-comparison"
)

In [ ]:
aiplatform.start_run("run_name")
aiplatform.log_metrics({...})
aiplatform.end_run()